# einops-rearrange-flatten — ex1: CNN flatten before the Linear head — channel + spatial collapse

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-rearrange-flatten`. Running the final beacon cell reports progress against the `Einops: Rearrange-as-flatten` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Rearrange-as-flatten` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-rearrange-flatten`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-rearrange-flatten"
DD_SUBTOPIC = "Einops: Rearrange-as-flatten"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Einops rearrange-as-flatten — quick refresher

The most common flatten pattern in computer vision:

```python
rearrange(x, 'b h w c -> b (h w c)')
rearrange(x, 'b c h w -> b (c h w)')
```

Parentheses *group* the named axes into a single composite axis. The order inside the parens is the **inner-loop-fastest** order of the flatten — change `(h w c)` to `(c h w)` and the resulting 1-D layout is completely different (even though the shape is the same).

**Why this is preferred over `.view(b, -1)`:**
1. **Self-documenting** — the axis names make the flatten order obvious.
2. **Order-explicit** — `view`/`flatten` use *whichever order the memory happens to be in*. If `x` is non-contiguous, `view` errors; `rearrange` transparently calls `.contiguous()` for you.
3. **Compositional** — `'b c h w -> b (c h w)'` is one of dozens of patterns you can mix and match without learning a separate API for each (`flatten`, `view`, `reshape`, `permute+view`, ...).

**Canonical CNN use.** Right before the final `Linear` head, `'b c h w -> b (c h w)'` collapses the spatial+channel axes into one feature axis. ARENA's `nn.Flatten` is exactly this pattern.

### Exercise 1 — CNN flatten before the Linear head — channel + spatial collapse

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `einops.rearrange` with the `'b c h w -> b (c h w)'` pattern to flatten a 4-D CNN feature map into a 2-D `(B, features)` matrix suitable for a `Linear` layer, while preserving the batch axis.
> Keywords: flatten, cnn, linear-head, axis-composition
> ```

**KCs targeted:** `rearrange-axis-composition-via-parens`, `rearrange-flatten-preserves-batch`

Implement `ex1_flatten_cnn_features(x)`.

- `x` has shape `(B, C, H, W)` — the canonical PyTorch CNN feature map (batch, channels, height, width).
- Return shape `(B, C*H*W)` — the per-sample feature vector, batch axis preserved.

**Constraint.** Use exactly `rearrange(x, 'b c h w -> b (c h w)')`. No `view`, no `flatten`, no `reshape`. The pattern string is load-bearing — the test asserts the flatten order matches `(c h w)`, which is `nn.Flatten`'s default and what `x.flatten(start_dim=1)` produces.

**Why this specific pattern.** ARENA's `SimpleMLP` and `ConvNet` both compose `Flatten() → Linear(...)`. The `Linear` layer expects a 2-D `(B, features)` input; this rearrange is the bridge.

In [ ]:
def ex1_flatten_cnn_features(x: Tensor) -> Tensor:
    """Flatten (B, C, H, W) -> (B, C*H*W) preserving batch axis."""
    raise NotImplementedError()


def _test_ex1():
    # Hand-build to verify the flatten order is (c h w).
    # Use distinguishable values so we can verify the ordering, not just the shape.
    x = t.arange(24.0).reshape(1, 2, 3, 4)  # (B=1, C=2, H=3, W=4)
    out = ex1_flatten_cnn_features(x)
    assert out.shape == (1, 24), f'expected (1, 24), got {tuple(out.shape)}'
    # rearrange 'b c h w -> b (c h w)' must produce the same byte order as x.flatten(1).
    expected = x.flatten(start_dim=1)
    assert t.equal(out, expected), (
        f'flatten order mismatch — did you write (h w c) or (w h c) instead of (c h w)?\n'
        f'got:      {out[0]}\n'
        f'expected: {expected[0]}'
    )

    # --- Realistic CNN-ish shape ---
    rng = t.Generator().manual_seed(0)
    B, C, H, W = 8, 32, 7, 7
    feat = t.randn(B, C, H, W, generator=rng)
    flat = ex1_flatten_cnn_features(feat)
    assert flat.shape == (B, C * H * W), f'expected ({B}, {C*H*W}), got {tuple(flat.shape)}'
    assert t.equal(flat, feat.flatten(start_dim=1)), 'must match nn.Flatten default behavior'

    # --- Pluggable into a real Linear layer ---
    lin = t.nn.Linear(C * H * W, 10)
    logits = lin(flat)
    assert logits.shape == (B, 10), 'output of Linear(flat) must be (B, num_classes)'

    # --- Edge: B=1 (single sample) ---
    single = t.randn(1, 4, 5, 5)
    fs = ex1_flatten_cnn_features(single)
    assert fs.shape == (1, 100)

    # --- Edge: C=H=W=1 (degenerate scalar-per-sample) ---
    scalar = t.randn(3, 1, 1, 1)
    ss = ex1_flatten_cnn_features(scalar)
    assert ss.shape == (3, 1)
    assert t.equal(ss, scalar.view(3, 1))
    print('flatten matches nn.Flatten on CNN-shaped, single-sample, and scalar inputs')
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_flatten_cnn_features(x: Tensor) -> Tensor:
    return rearrange(x, 'b c h w -> b (c h w)')
```

**Why `(c h w)` and not `(h w c)`.** PyTorch's default tensor layout is channels-first (`B, C, H, W`). The byte order in memory is `c` outer, `w` inner. `(c h w)` matches that — it's free, no copy needed (einops doesn't even call `.contiguous()` in this case). `(h w c)` would force a permute + copy.

**Equivalent torch idioms.**
- `x.flatten(start_dim=1)` — most concise.
- `x.view(x.shape[0], -1)` — fastest (requires contiguous).
- `nn.Flatten()(x)` — the Module form ARENA uses inside `Sequential`.

All four produce the identical tensor; the test checks against `x.flatten(start_dim=1)` as the ground truth.

**Why we prefer the rearrange form even though it's the longest.** Reading code: `'b c h w -> b (c h w)'` tells you the axes by name, the order they're flattened in, and what survives. `x.view(B, -1)` tells you nothing about which axes get collapsed or in what order. In a CNN pipeline with multiple reshape steps (patchify, unpatchify, channel-shuffle, ...), named patterns prevent silent bugs.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()